# Transformation explorer

Plots a TIMIT WAV file's raw waveform next to its spectrogram, featurized like `src/debug_items.py` does
(`compute_mel_spectrogram` / `compute_linear_spectrogram` from `src/data/features.py`, then `apply_drc`).
The widgets start at the `TransformationConfig` defaults; changing any of them (or the WAV file) re-does the transformation.

The second row shows what a model is fed:

- **Normalized utterance.** The whole utterance after `featurize_waveform`, under the selected `normalization`:
  - `frame`: per-frame z-score. This also removes each frame's level, i.e. the energy contour.
  - `segment`: each band's mean over the drawn segment is subtracted, which removes gain and channel but keeps the energy contour.
  - `none`: no normalization.

  The frames the drawn segment uses are marked.
- **Drawn segment.** One segment of `segment_duration`, drawn with the selected strategy from `DRAW_STRATEGIES` (`src/data/segments.py`), shown exactly as the model receives it, so `segment` normalization is applied here:
  - OS, SS, SU, MIX or RANDBS.
  - BS{b}: an OS crop cut into blocks of b frames, the block order permuted, the order inside each block kept.
  - GO{b} (the SST bracket's control C_b): blocks of b frames with a gap of `gap ratio` × b frames (`EvaluationConfig.sst_gap_ratio`) between them, kept in their original order. The marks on the normalized utterance show the gaps between the blocks.
  - GS{b} (S_b): GO{b}'s blocks in random order. With the same seed, GO{b} and GS{b} take the same blocks, so switching between them shows only the reordering.

  The seed sets the random position, as for a training draw. Evaluation (`_eval_windows`) instead places its windows at evenly spaced positions.
- **Shuffle pattern.** Which original frame each segment position comes from. Each block of BS, GO or GS appears as a rising run of points. GO's runs rise from left to right with jumps between them (the gaps); BS's and GS's runs are in a different order.

In [1]:
import html
import sys
from dataclasses import replace
from functools import partial
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import soundfile as sf
from IPython.display import Audio, display
from plotly.subplots import make_subplots
import plotly.io as pio; pio.renderers.default = "notebook"

ROOT = Path.cwd().parent  # notebook runs from doc/
sys.path.insert(0, str(ROOT))

from src.config import DataConfig, EvaluationConfig, TransformationConfig
from src.data.dataset import featurize_waveform
from src.data.features import apply_drc, compute_linear_spectrogram, compute_mel_spectrogram
from src.data.segments import DRAW_STRATEGIES, gapped_span

In [2]:
timit_root = Path("~/.cache/datasets/timit").expanduser()
# Labelled relative to the TIMIT/ folder, e.g. "TRAIN/DR1/FCJF0/SA1.WAV".
wav_paths = [(str(p.relative_to(p.parents[3])), p) for p in sorted(timit_root.rglob("*")) if p.suffix.lower() == ".wav"]
len(wav_paths)

6300

In [ ]:
defaults = TransformationConfig()
wav_path = widgets.Dropdown(options=wav_paths, description="WAV file", layout=widgets.Layout(width="500px"))
frame_length_s = widgets.FloatText(value=defaults.frame_length_s, step=0.001, description="frame_length_s")
frame_step_s = widgets.FloatText(value=defaults.frame_step_s, step=0.001, description="frame_step_s")
nfft = widgets.IntText(value=defaults.nfft, description="nfft")
n_mels = widgets.IntText(value=defaults.n_mels, description="n_mels")
fmin = widgets.BoundedFloatText(value=defaults.fmin, min=0, max=defaults.sample_rate, description="fmin")
fmax = widgets.BoundedFloatText(value=defaults.fmax, min=defaults.fmin, max=defaults.sample_rate, description="fmax")
type_ = widgets.Dropdown(options=[("mel", "mel"), ("lin", "linear")], value=defaults.type, description="type")
window = widgets.Dropdown(options=["hann", "hamming"], value=defaults.window, description="window")
normalization = widgets.Dropdown(options=["frame", "segment", "none"], value=defaults.normalization, description="normalization")
strategy = widgets.Dropdown(options=list(DRAW_STRATEGIES), value="OS", description="strategy")
segment_duration = widgets.BoundedFloatText(value=DataConfig().segment_duration, min=0.1, step=0.1, description="segment [s]")
seed = widgets.IntText(value=0, description="seed")
gap_ratio = widgets.BoundedFloatText(value=EvaluationConfig().sst_gap_ratio, min=0, step=0.25, description="gap ratio")

# Keep fmin <= fmax: each bounds the other.
widgets.dlink((fmin, "value"), (fmax, "min"))
widgets.dlink((fmax, "value"), (fmin, "max"))


# The figure (as an iframe) and the audio player go into HTML widgets: plotly's and Audio's own
# notebook outputs don't render inside an Output widget in PyCharm.
plot = widgets.HTML()
player = widgets.HTML()


def show(wav_path, frame_length_s, frame_step_s, nfft, n_mels, fmin, fmax, type_, window,
         normalization, strategy, segment_duration, seed, gap_ratio):
    waveform, sample_rate = sf.read(str(wav_path), dtype="float32")
    assert sample_rate == defaults.sample_rate, f"{wav_path} is {sample_rate} Hz, config expects {defaults.sample_rate} Hz"
    config = replace(defaults, frame_length_s=frame_length_s, frame_step_s=frame_step_s, nfft=nfft,
                     n_mels=n_mels, fmin=fmin, fmax=fmax, type=type_, window=window, normalization=normalization)
    # FFT magnitude of every frame (the linear spectrogram); the mel spectrogram is computed from it.
    stft = apply_drc(compute_linear_spectrogram(waveform, config))
    if type_ == "linear":
        features = stft
    else:
        features = apply_drc(compute_mel_spectrogram(waveform, config))

    # What the model is fed: the utterance as featurize_waveform normalizes it ("frame" only), then one
    # segment drawn with `strategy`. Drawing from an array of frame indices with an identically seeded
    # Generator makes the same random choices, which yields the segment's source frames.
    normalized = featurize_waveform(waveform, config)
    segment_length = DataConfig(segment_duration=segment_duration).segment_length(config)
    if normalized.shape[0] <= segment_length:
        plot.value = f"<b>The utterance ({normalized.shape[0]} frames) must be longer than one segment ({segment_length} frames).</b>"
        player.value = ""
        return
    draw = DRAW_STRATEGIES[strategy]
    if strategy.startswith(("GO", "GS")):
        # As trainer._eval_windows converts evaluation.sst_gap_ratio.
        gap = round(gap_ratio * int(strategy[2:]))
        span = gapped_span(segment_length, int(strategy[2:]), gap)
        if normalized.shape[0] < span:
            plot.value = f"<b>The utterance ({normalized.shape[0]} frames) is shorter than {strategy}'s gapped span ({span} frames).</b>"
            player.value = ""
            return
        draw = partial(draw, gap=gap)
    source_frames = draw(np.arange(normalized.shape[0])[:, None], segment_length, np.random.default_rng(seed))[:, 0]
    segment = draw(normalized, segment_length, np.random.default_rng(seed))
    if normalization == "segment":
        # As src/models/registry.py's SegmentMeanNormalization does to every model input. The utterance gets the
        # same band means subtracted, so the drawn frames have the same values in both panels.
        band_means = segment.mean(axis=0, keepdims=True)
        segment = segment - band_means
        normalized = normalized - band_means

    # Plotly, so hovering shows the exact values at the mouse position.
    title = f"waveform: {wav_path.relative_to(wav_path.parents[3])}"
    freq_label = "mel bin" if type_ == "mel" else "freq bin"
    frame_times = np.arange(features.shape[0]) * config.frame_step_s
    fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.1, vertical_spacing=0.16, subplot_titles=[
        title, f"FFT per frame: {stft.shape[0]} frames x {stft.shape[1]} bins",
        f"{type_} spectrogram {features.shape[0]} frames x {features.shape[1]} bins",
        f"normalized ({normalization}), frames of the drawn segment marked",
        f"{strategy} segment as the model sees it: {segment.shape[0]} frames",
        "shuffle pattern: source frame per segment position"])

    def colorbar(row, col, title):
        """A colour bar right next to subplot (row, col), as tall as it."""
        subplot = fig.get_subplot(row, col)
        (_, x1), (y0, y1) = subplot.xaxis.domain, subplot.yaxis.domain
        return dict(title_text=title, x=x1 + 0.005, xanchor="left", y=(y0 + y1) / 2, yanchor="middle", len=y1 - y0,
                    thickness=12)

    fig.add_trace(go.Scattergl(
        x=np.arange(len(waveform)) / sample_rate, y=waveform, mode="lines", line_width=0.5, showlegend=False,
        hovertemplate="time: %{x:.4f} s<br>amplitude: %{y:.4f}<extra></extra>",
    ), row=1, col=1)
    fig.add_trace(go.Heatmap(
        z=stft.T, x=frame_times, y=np.arange(stft.shape[1]) * sample_rate / config.nfft,
        colorscale="Magma", colorbar=colorbar(1, 2, "DRC level"),
        hovertemplate="time: %{x:.3f} s<br>frequency: %{y:.1f} Hz<br>DRC level: %{z:.2f}<extra></extra>",
    ), row=1, col=2)
    fig.add_trace(go.Heatmap(
        z=features.T, x=frame_times, y=np.arange(features.shape[1]),
        colorscale="Magma", colorbar=colorbar(1, 3, "DRC level"),
        hovertemplate=f"time: %{{x:.3f}} s<br>{freq_label}: %{{y}}<br>DRC level: %{{z:.2f}}<extra></extra>",
    ), row=1, col=3)
    # Both normalized heatmaps share one colour scale, so equal values get equal colours. Normalized features
    # are centred on 0 (diverging colours); without normalization they're DRC levels.
    level = dict(colorscale="Magma") if normalization == "none" else dict(colorscale="RdBu_r", cmid=0)
    level_name = "DRC level" if normalization == "none" else "normalized level"
    fig.update_layout(coloraxis=dict(**level, colorbar=colorbar(2, 2, level_name)))
    fig.add_trace(go.Heatmap(
        z=normalized.T, x=frame_times, y=np.arange(normalized.shape[1]), coloraxis="coloraxis",
        hovertemplate=f"time: %{{x:.3f}} s<br>{freq_label}: %{{y}}<br>level: %{{z:.2f}}<extra></extra>",
    ), row=2, col=1)
    fig.add_trace(go.Scattergl(
        x=np.unique(source_frames) * config.frame_step_s, y=np.full(len(np.unique(source_frames)), -2),
        mode="markers", marker=dict(color="lime", size=4, symbol="line-ns-open"), showlegend=False,
        hovertemplate="drawn frame at %{x:.3f} s<extra></extra>",
    ), row=2, col=1)
    fig.add_trace(go.Heatmap(
        z=segment.T, x=np.arange(segment.shape[0]), y=np.arange(segment.shape[1]), coloraxis="coloraxis",
        customdata=np.tile(source_frames, (segment.shape[1], 1)),
        hovertemplate=f"position: %{{x}}<br>source frame: %{{customdata}}<br>{freq_label}: %{{y}}<br>level: %{{z:.2f}}<extra></extra>",
    ), row=2, col=2)
    fig.add_trace(go.Scattergl(
        x=np.arange(len(source_frames)), y=source_frames, mode="markers", marker=dict(size=4), showlegend=False,
        hovertemplate="position: %{x}<br>source frame: %{y}<extra></extra>",
    ), row=2, col=3)
    fig.update_xaxes(title_text="segment position [frame]", row=2, col=2)
    fig.update_xaxes(title_text="segment position [frame]", row=2, col=3)
    fig.update_yaxes(title_text=freq_label, row=2, col=1)
    fig.update_yaxes(title_text=freq_label, row=2, col=2)
    fig.update_yaxes(title_text="source frame", row=2, col=3)
    if type_ == "mel":
        # The part of the FFT spectrum the mel filterbank spreads its filters over.
        for name, f in [("fmin", fmin), ("fmax", fmax)]:
            fig.add_hline(y=f, line=dict(color="cyan", width=1, dash="dash"), annotation_text=name,
                          annotation_font_color="cyan", row=1, col=2)
    fig.update_xaxes(title_text="time [s]", row=1)
    fig.update_xaxes(title_text="time [s]", row=2, col=1)
    fig.update_yaxes(title_text="amplitude", row=1, col=1)
    fig.update_yaxes(title_text="frequency [Hz]", row=1, col=2)
    fig.update_yaxes(title_text=freq_label, row=1, col=3)
    fig.update_layout(height=800, margin=dict(l=60, r=80, t=40, b=50))
    page = fig.to_html(include_plotlyjs="cdn", config=dict(responsive=True))
    plot.value = f'<iframe srcdoc="{html.escape(page)}" style="width: 100%; height: 820px; border: 0"></iframe>'
    player.value = Audio(waveform, rate=sample_rate)._repr_html_()


output = widgets.interactive_output(show, dict(
    wav_path=wav_path, frame_length_s=frame_length_s, frame_step_s=frame_step_s, nfft=nfft, n_mels=n_mels,
    fmin=fmin, fmax=fmax, type_=type_, window=window,
    normalization=normalization, strategy=strategy, segment_duration=segment_duration, seed=seed, gap_ratio=gap_ratio,
))
display(widgets.VBox([
    wav_path,
    widgets.HBox([frame_length_s, frame_step_s, nfft, n_mels]),
    widgets.HBox([fmin, fmax, type_, window]),
    widgets.HBox([normalization, strategy, segment_duration, seed, gap_ratio]),
    plot,
    player,
    output,
]))


## OC, GC, GO and GS of a VoxCeleb1-H utterance

How the SST bracket's gapped test inputs are cut from one utterance of the VoxCeleb1-H trial list (`list_test_hard2.txt`, the reported SV of the `pretrained-*-voxceleb-*` configs). Each input is built by `trainer._eval_windows` exactly as in evaluation: one window per utterance, its span of `gapped_span(segment, b, g)` frames centred in the utterance (position 0.5), with g = round(`gap ratio` × b) frames. The front-end sets the frame grid the inputs are cut on: WeSpeaker's fbank (ResNet34, 10 ms frames) or w2v-BERT 2.0's stacked fbank (20 ms frames).

- **Utterance (top).** The featurized utterance. The span lies between the black lines and GO/GS's gaps are shaded. The rugs below the spectrogram show which frames each input takes.
- **The four inputs (middle)**, as the model receives them, each with `segment` frames. Dotted lines mark joins, i.e. two frames meeting that weren't neighbours in the utterance:
  - OC{b}: the span's contiguous middle. An OS window of as many frames, at the same place and on the same trials. It has no join.
  - GC{b}: the span's first half-segment and its last, with all of GO's gap frames removed as one cut, so it has a single join.
  - GO{b}: blocks of b frames with g frames skipped between them, in their original order.
  - GS{b}: GO{b}'s blocks in a random order (`seed`), with no block left in place. These are the same blocks as GO{b}, only reordered.
- **Source frame per position (bottom).** The time in the utterance that each input position comes from.
- **Players.** The first plays the whole utterance. The four below it play OC, GC, GO and GS: each input position contributes the frame period (10 or 20 ms) of samples starting where its source frame starts. Contiguous runs therefore sound like the original, and every join is an audible hard cut.

An utterance shorter than the span has no such window, and its trials aren't scored for that b.

In [4]:
from src.data.segments import BLOCK_SIZES
from src.training.trainer import _eval_windows

vox1_root = Path("~/.cache/datasets/voxceleb1").expanduser()
# The utterances of the VoxCeleb1-H trial list (voxceleb.eval_trial_meta_url), on which the SST runs report SV.
with open(vox1_root / "list_test_hard2.txt") as f:
    vox1h_paths = sorted({path for line in f for path in line.split()[1:]})
vox1h_by_speaker = {}
for path in vox1h_paths:
    vox1h_by_speaker.setdefault(path.split("/")[0], []).append(path)

# The pretrained models' first frame grids, what the draws cut (configs/pretrained-*-voxceleb-2s.yaml).
sst_front_ends = {"wespeaker_fbank (ResNet34, 10 ms frames)": "wespeaker_fbank", "w2vbert (w2v-BERT 2.0, 20 ms frames)": "w2vbert"}
sst_strategies = ("OC", "GC", "GO", "GS")

sst_speaker = widgets.Dropdown(options=sorted(vox1h_by_speaker), description="speaker")
sst_utterance = widgets.Dropdown(options=vox1h_by_speaker[sst_speaker.value], description="utterance",
                                 layout=widgets.Layout(width="400px"))
sst_speaker.observe(lambda change: setattr(sst_utterance, "options", vox1h_by_speaker[change.new]), names="value")
sst_front_end = widgets.Dropdown(options=sst_front_ends, description="front-end", layout=widgets.Layout(width="400px"))
sst_block_size = widgets.Dropdown(options=BLOCK_SIZES, value=8, description="block b")
sst_gap_ratio = widgets.BoundedFloatText(value=1.25, min=0, step=0.25, description="gap ratio")  # the pretrained 2s configs'
sst_segment_duration = widgets.BoundedFloatText(value=2.0, min=0.1, step=0.1, description="segment [s]")
sst_seed = widgets.IntText(value=0, description="seed (GS)")

sst_plot = widgets.HTML()
sst_player = widgets.HTML()
# One player per input: what each of OC/GC/GO/GS sounds like.
sst_input_players = {name: widgets.HTML() for name in sst_strategies}


def show_sst(utterance, front_end, block_size, gap_ratio, segment_duration, seed):
    waveform, sample_rate = sf.read(str(vox1_root / "wav" / utterance), dtype="float32")
    config = TransformationConfig(type=front_end, frame_length_s=0.025, frame_step_s=0.01)
    assert sample_rate == config.sample_rate, f"{utterance} is {sample_rate} Hz, config expects {config.sample_rate} Hz"
    features = featurize_waveform(waveform, config)
    period = config.frame_period_s
    segment_length = DataConfig(segment_duration=segment_duration).segment_length(config)
    gap = round(gap_ratio * block_size)  # as trainer._eval_windows converts evaluation.sst_gap_ratio
    span = gapped_span(segment_length, block_size, gap)
    sst_player.value = Audio(waveform, rate=sample_rate)._repr_html_()
    if features.shape[0] < span:
        for input_player in sst_input_players.values():
            input_player.value = ""
        sst_plot.value = (f"<b>The utterance ({features.shape[0]} frames) is shorter than the gapped span ({span} frames): "
                          f"it has no OC/GC/GO/GS window and its trials aren't scored for b = {block_size}.</b>")
        return

    # Each input exactly as the evaluation draws it (one window in the middle of the utterance). Drawing from an
    # array of frame indices with an identically seeded Generator yields each input's source frames.
    inputs, sources = {}, {}
    for name in sst_strategies:
        strategy = f"{name}{block_size}"
        inputs[name] = _eval_windows(features, segment_length, segment_length, 1, strategy,
                                     np.random.default_rng(seed), gap_ratio)[0]
        sources[name] = _eval_windows(np.arange(features.shape[0])[:, None], segment_length, segment_length, 1,
                                      strategy, np.random.default_rng(seed), gap_ratio)[0][:, 0]

    # Each input as audio: every source frame contributes the frame period of samples starting where that frame
    # starts, so a contiguous run plays the original signal and each join is a hard cut. normalize=False keeps the
    # four players at the utterance's level, so they're comparable.
    hop = round(period * sample_rate)
    for name in sst_strategies:
        audio = waveform[(sources[name][:, None] * hop + np.arange(hop)).ravel()]
        sst_input_players[name].value = (f"<b>{name}{block_size}</b> ({len(audio) / sample_rate:.2f} s)"
                                         + Audio(audio, rate=sample_rate, normalize=False)._repr_html_())

    num_blocks = -(-segment_length // block_size)
    start = round(0.5 * (features.shape[0] - span))  # src/data/segments.py's position 0.5
    cut = (num_blocks - 1) * gap
    fig = make_subplots(
        rows=3, cols=4, specs=[[{"colspan": 4}, None, None, None], [{}] * 4, [{}] * 4],
        row_heights=[0.4, 0.35, 0.25], vertical_spacing=0.09, horizontal_spacing=0.05, subplot_titles=[
            f"{utterance}: {features.shape[0]} frames; span of {span} frames ({span * period:.2f} s) "
            f"in its middle, GO/GS gaps shaded (GO joins splice frames {(gap + 1) * period * 1000:.0f} ms apart)<br>"
            "frames each input takes, below: OC (blue), GC (red), GO/GS (green)",
            f"OC{block_size}: the span's contiguous middle",
            f"GC{block_size}: one cut of {cut} frames",
            f"GO{block_size}: {num_blocks} blocks, gaps of {gap} frames",
            f"GS{block_size}: GO's blocks shuffled",
            *[f"{name}{block_size}: source frame per position" for name in sst_strategies]])

    # One colour scale for the utterance and all inputs, so equal values get equal colours.
    fig.update_layout(coloraxis=dict(colorscale="RdBu_r", cmid=0, colorbar=dict(title_text="level", len=0.4, y=0.8)))
    fig.add_trace(go.Heatmap(
        z=features.T, x=np.arange(features.shape[0]) * period, y=np.arange(features.shape[1]), coloraxis="coloraxis",
        hovertemplate="time: %{x:.3f} s<br>bin: %{y}<br>level: %{z:.2f}<extra></extra>",
    ), row=1, col=1)
    for i in range(num_blocks - 1):
        gap_start = start + i * (block_size + gap) + block_size
        fig.add_vrect(x0=(gap_start - 0.5) * period, x1=(gap_start + gap - 0.5) * period, fillcolor="black",
                      opacity=0.35, line_width=0, row=1, col=1)
    for x in (start, start + span):
        fig.add_vline(x=(x - 0.5) * period, line=dict(color="black", width=1.5), row=1, col=1)
    # Below the spectrogram: a rug of the frames each input takes.
    colors = dict(OC="#1f77b4", GC="#d62728", GO="#2ca02c", GS="#2ca02c")
    for offset, name in enumerate(("OC", "GC", "GO")):
        taken = np.unique(sources[name])
        fig.add_trace(go.Scattergl(
            x=taken * period, y=np.full(len(taken), -4 - 6 * offset), mode="markers", showlegend=False,
            marker=dict(color=colors[name], size=5, symbol="line-ns-open"),
            hovertemplate=f"{name}: frame at %{{x:.3f}} s<extra></extra>",
        ), row=1, col=1)

    for col, name in enumerate(sst_strategies, start=1):
        segment, source = inputs[name], sources[name]
        fig.add_trace(go.Heatmap(
            z=segment.T, x=np.arange(segment.shape[0]), y=np.arange(segment.shape[1]), coloraxis="coloraxis",
            customdata=np.tile(source * period, (segment.shape[1], 1)),
            hovertemplate="position: %{x}<br>from: %{customdata:.3f} s<br>bin: %{y}<br>level: %{z:.2f}<extra></extra>",
        ), row=2, col=col)
        # Joins: where two frames meet that weren't neighbours in the utterance.
        for position in np.flatnonzero(np.diff(source) != 1) + 1:
            fig.add_vline(x=position - 0.5, line=dict(color="black", width=1, dash="dot"), row=2, col=col)
        fig.add_trace(go.Scattergl(
            x=np.arange(len(source)), y=source * period, mode="markers", marker=dict(color=colors[name], size=3),
            showlegend=False, hovertemplate="position: %{x}<br>from: %{y:.3f} s<extra></extra>",
        ), row=3, col=col)
        fig.update_xaxes(title_text="input position [frame]", row=3, col=col)
    fig.update_xaxes(title_text="time [s]", row=1, col=1)
    fig.update_yaxes(title_text="bin", row=1, col=1)
    fig.update_yaxes(title_text="bin", row=2, col=1)
    fig.update_yaxes(title_text="source time [s]", row=3, col=1)
    fig.update_layout(height=1000, margin=dict(l=60, r=80, t=60, b=50))
    page = fig.to_html(include_plotlyjs="cdn", config=dict(responsive=True))
    sst_plot.value = f'<iframe srcdoc="{html.escape(page)}" style="width: 100%; height: 1020px; border: 0"></iframe>'


sst_output = widgets.interactive_output(show_sst, dict(
    utterance=sst_utterance, front_end=sst_front_end, block_size=sst_block_size, gap_ratio=sst_gap_ratio,
    segment_duration=sst_segment_duration, seed=sst_seed,
))
display(widgets.VBox([
    widgets.HBox([sst_speaker, sst_utterance, sst_front_end]),
    widgets.HBox([sst_block_size, sst_gap_ratio, sst_segment_duration, sst_seed]),
    sst_plot,
    sst_player,
    widgets.HBox(list(sst_input_players.values())),
    sst_output,
]))